# Wongnai Thai Restaurant Reviews — Clause-Level & Contrast-Aware NLP Study

**Dataset:** [Wongnai/wongnai_reviews](https://huggingface.co/datasets/Wongnai/wongnai_reviews) (`500` balanced Thai reviews across `1–5` stars)

## Core Conference Idea
Standard document-level embeddings (`E5-Base`) compress an entire multi-paragraph Thai review into a single vector (and truncate at 512 tokens), causing mixed praise + complaints (*"อาหารอร่อย... แต่บริการแย่"*) to blur together. We propose and evaluate a **Clause-Level + Contrast-Aware Pipeline**:

1. **RQ1 (Document-Level vs. Clause/Contrast-Aware Pooling):** Does splitting Thai reviews into clauses (`mean + max + contrast-tail` pooling) fix `E5-Base` on long and `"แต่/เสียดาย"` contrast reviews compared to `TF-IDF` and `Whole-Review E5`?
2. **RQ2 (Noise-Filtered Complaint Clustering):** Does clustering **extracted complaint clauses** (with Thai stopwords removed) produce cleaner, higher-Silhouette pain-point themes than clustering raw full reviews?
3. **RQ3 (Document-Level vs. Clause-Max Aspect Detection):** Does **Clause-Level Max-Similarity** fix the dilution problem of Document-Level Zero-Shot `E5-Base` on Thai aspect detection (`Food`, `Service`, `Price`, `Cleanliness`)?


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# Fix Thai font rendering in matplotlib
plt.rcParams['font.sans-serif'] = ['Tahoma', 'Leelawadee UI', 'Angsana New', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

ROOT = Path('.').resolve()
DATA_PATH = ROOT / 'data' / 'wongnai_500.csv'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['char_len'] = df['review_body'].astype(str).str.len()
df['len_bucket'] = pd.cut(df['char_len'], bins=[0, 250, 600, 100000], labels=['Short (<=250)', 'Medium (251-600)', 'Long (>600)'])

print('Loaded dataset:', df.shape)
display(df['sentiment'].value_counts().to_frame('count'))
display(df['len_bucket'].value_counts().sort_index().to_frame('count'))


## 1. Thai Tokenization (Stopwords Removed) & Clause Segmentation


In [ ]:
TH_STOPWORDS = set(thai_stopwords()) - {'ไม่', 'น้อย', 'แพง', 'ช้า', 'นาน', 'ดี', 'แย่', 'อร่อย'}

def thai_tokenize_clean(text: str) -> list[str]:
    tokens = word_tokenize(str(text), engine='newmm')
    return [t.strip() for t in tokens if len(t.strip()) > 1 and t.strip() not in TH_STOPWORDS and not t.strip().isdigit()]

def split_thai_clauses(text: str, max_clauses: int = 10) -> list[str]:
    # Split on newlines and contrast markers while keeping clauses meaningful
    raw_parts = re.split(r'[\n]+|(?=(?:แต่|เสียดาย|ยกเว้น|ติดตรง))', str(text))
    clauses = []
    for p in raw_parts:
        p = p.strip()
        if len(p) >= 8:
            # Further split very long paragraphs by spaces if needed
            if len(p) > 220:
                sub = [s.strip() for s in p.split('  ') if len(s.strip()) >= 8]
                clauses.extend(sub if sub else [p])
            else:
                clauses.append(p)
    if not clauses:
        clauses = [str(text).strip()]
    return clauses[:max_clauses]

df['clauses'] = df['review_body'].apply(split_thai_clauses)
df['num_clauses'] = df['clauses'].apply(len)
print('Average clauses per review:', round(df['num_clauses'].mean(), 2))

tfidf = TfidfVectorizer(tokenizer=thai_tokenize_clean, token_pattern=None, min_df=2, max_features=3000)
X_tfidf = tfidf.fit_transform(df['review_body'])

e5_model = SentenceTransformer('intfloat/multilingual-e5-base')

# 1) Whole-review E5 embedding (baseline)
X_e5_doc = e5_model.encode(
    ['passage: ' + str(t) for t in df['review_body']],
    normalize_embeddings=True,
    show_progress_bar=True,
)

# 2) Clause-level E5 embeddings (encode all unique clauses in batch)
all_clauses = [c for clist in df['clauses'] for c in clist]
clause_vecs_flat = e5_model.encode(
    ['passage: ' + c for c in all_clauses],
    normalize_embeddings=True,
    show_progress_bar=True,
)

clause_vecs_per_review = []
X_e5_chunked = []
idx = 0
for clist in df['clauses']:
    n = len(clist)
    vecs = clause_vecs_flat[idx:idx+n]
    idx += n
    clause_vecs_per_review.append(vecs)
    mean_v = vecs.mean(axis=0)
    # Contrast/tail emphasis: average of clauses containing contrast words or final clause
    contrast_idx = [j for j, c in enumerate(clist) if any(w in c for w in ['แต่', 'เสียดาย', 'ยกเว้น', 'ติดตรง', 'ไม่ประทับใจ', 'แพง', 'แย่'])]
    tail_v = vecs[contrast_idx].mean(axis=0) if contrast_idx else vecs[-1]
    combined = np.concatenate([mean_v, tail_v])
    combined = combined / (np.linalg.norm(combined) + 1e-12)
    X_e5_chunked.append(combined)

X_e5_chunked = np.vstack(X_e5_chunked)
print('Shapes -> TF-IDF:', X_tfidf.shape, '| Whole E5:', X_e5_doc.shape, '| Clause+Contrast E5:', X_e5_chunked.shape)


## 2. RQ1: Whole-Review vs. Clause+Contrast E5 Across Review Lengths & Contrast Slices


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y = df['sentiment']

df['pred_tfidf'] = cross_val_predict(LogisticRegression(max_iter=1000, random_state=42), X_tfidf, y, cv=cv)
df['pred_e5_doc'] = cross_val_predict(LogisticRegression(max_iter=1000, random_state=42), X_e5_doc, y, cv=cv)
df['pred_e5_clause'] = cross_val_predict(LogisticRegression(max_iter=1000, random_state=42), X_e5_chunked, y, cv=cv)

models = [
    ('TF-IDF (PyThaiNLP)', 'pred_tfidf'),
    ('E5-Base (Whole Review)', 'pred_e5_doc'),
    ('E5-Base (Clause + Contrast)', 'pred_e5_clause'),
]
slices = [
    ('All Reviews (n=500)', pd.Series(True, index=df.index)),
    ('No Contrast Word', ~df['has_contrast_word']),
    ('Has Contrast Word', df['has_contrast_word']),
    ('Long Reviews (>600 chars)', df['len_bucket'] == 'Long (>600)'),
]

rq1_rows = []
for mname, col in models:
    for sname, mask in slices:
        sub = df[mask]
        rq1_rows.append({
            'Model': mname,
            'Slice': sname,
            'N': len(sub),
            'Accuracy': round(accuracy_score(sub['sentiment'], sub[col]), 3),
            'Macro_F1': round(f1_score(sub['sentiment'], sub[col], average='macro'), 3),
        })

rq1_df = pd.DataFrame(rq1_rows)
display(rq1_df.pivot(index='Slice', columns='Model', values='Macro_F1').loc[[s[0] for s in slices]])

fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(slices))
w = 0.25
palette = ['#28537A', '#E07A5F', '#2A9D8F']
for i, ((mname, _), col) in enumerate(zip(models, palette)):
    vals = [rq1_df[(rq1_df['Model'] == mname) & (rq1_df['Slice'] == s[0])]['Macro_F1'].iloc[0] for s in slices]
    bars = ax.bar(x + (i - 1) * w, vals, width=w, label=mname, color=col, edgecolor='black', linewidth=0.6)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([s[0] for s in slices], fontsize=9.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Macro F1', fontweight='bold')
ax.set_title('RQ1: Whole-Review E5 vs. Clause+Contrast E5 on Thai Reviews', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig1_rq1_sentiment_contrast.png', dpi=180, bbox_inches='tight')
plt.show()


## 3. RQ2: Raw-Review vs. Complaint-Clause Clustering (with Thai Stopwords Removed)


In [ ]:
neg_mask = (df['sentiment'] == 'Negative').values
neg_df = df[neg_mask].copy().reset_index(drop=True)

# Pick the most negative/contrastive clause per negative review
NEG_CUES = ['ไม่', 'แพง', 'ช้า', 'นาน', 'แย่', 'คาว', 'จืด', 'เค็ม', 'สกปรก', 'ผิดหวัง', 'เสียดาย', 'แต่', 'น้อย', 'หัก']
def pick_complaint_clause(clist: list[str]) -> str:
    scored = sorted(clist, key=lambda c: sum(w in c for w in NEG_CUES), reverse=True)
    return scored[0]

neg_df['complaint_clause'] = neg_df['clauses'].apply(pick_complaint_clause)

X_neg_raw_tfidf = X_tfidf[neg_mask]
X_neg_raw_e5 = X_e5_doc[neg_mask]
X_neg_clause_e5 = e5_model.encode(['passage: ' + c for c in neg_df['complaint_clause']], normalize_embeddings=True)

k = 4
km_raw_tfidf = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_neg_raw_tfidf)
km_raw_e5 = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_neg_raw_e5)
km_clause_e5 = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_neg_clause_e5)

sil_raw_tfidf = silhouette_score(X_neg_raw_tfidf, km_raw_tfidf.labels_)
sil_raw_e5 = silhouette_score(X_neg_raw_e5, km_raw_e5.labels_)
sil_clause_e5 = silhouette_score(X_neg_clause_e5, km_clause_e5.labels_)

print(f'Silhouette (k={k}) -> Raw TF-IDF: {sil_raw_tfidf:.4f} | Raw E5: {sil_raw_e5:.4f} | Complaint-Clause E5: {sil_clause_e5:.4f}')

neg_df['cluster_clause_e5'] = km_clause_e5.labels_
clause_tfidf = TfidfVectorizer(tokenizer=thai_tokenize_clean, token_pattern=None, min_df=2, max_features=1500)
X_c_tfidf = clause_tfidf.fit_transform(neg_df['complaint_clause'])
c_terms = np.array(clause_tfidf.get_feature_names_out())

print('\nTop Thai complaint keywords per Complaint-Clause E5 cluster:')
for c in range(k):
    idx_c = (neg_df['cluster_clause_e5'] == c).values
    mean_w = np.asarray(X_c_tfidf[idx_c].mean(axis=0)).ravel()
    top_words = [c_terms[j] for j in mean_w.argsort()[::-1][:8]]
    print(f'Cluster {c} (n={idx_c.sum()}): {top_words}')

coords = PCA(n_components=2, random_state=42).fit_transform(X_neg_clause_e5)
fig, ax = plt.subplots(figsize=(8.5, 5.5))
colors = ['#28537A', '#E07A5F', '#2A9D8F', '#E9C46A']
for c in range(k):
    m = neg_df['cluster_clause_e5'] == c
    ax.scatter(coords[m, 0], coords[m, 1], label=f'Cluster {c} (n={m.sum()})', alpha=0.85, s=48, color=colors[c], edgecolors='black', linewidth=0.4)
ax.set_title(f'RQ2: Complaint-Clause E5 Clusters (Silhouette={sil_clause_e5:.3f} vs Raw E5={sil_raw_e5:.3f})', fontweight='bold')
ax.set_xlabel('PCA 1')
ax.set_ylabel('PCA 2')
ax.grid(alpha=0.3, linestyle='--')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig2_rq2_complaint_clusters.png', dpi=180, bbox_inches='tight')
plt.show()


## 4. RQ3: Document-Level vs. Clause-Max Aspect Detection


In [ ]:
ASPECT_KEYWORDS = {
    'Food_Taste': ['อร่อย', 'รสชาติ', 'จืด', 'เค็ม', 'หวาน', 'เผ็ด', 'คาว', 'สด', 'ไหม้', 'เมนู'],
    'Service_Wait': ['บริการ', 'พนักงาน', 'เสิร์ฟ', 'รอ', 'ช้า', 'นาน', 'คิว', 'พูดจา', 'หน้าบึ้ง'],
    'Price_Value': ['ราคา', 'แพง', 'บาท', 'คุ้ม', 'ปริมาณ', 'น้อย', 'ถูก', 'เกินไป'],
    'Cleanliness_Ambience': ['สะอาด', 'สกปรก', 'แมลง', 'ห้องน้ำ', 'บรรยากาศ', 'ร้าน', 'โต๊ะ', 'ที่จอดรถ', 'แคบ'],
}

ASPECT_PROMPTS = {
    'Food_Taste': 'query: รสชาติอาหารอร่อยหรือไม่อร่อย คุณภาพอาหารและเมนู',
    'Service_Wait': 'query: การบริการของพนักงาน ความช้าเร็วในการเสิร์ฟและการรอคิว',
    'Price_Value': 'query: ราคาอาหารแพงหรือถูก ความคุ้มค่าและปริมาณอาหาร',
    'Cleanliness_Ambience': 'query: ความสะอาดของร้าน สุขอนามัย บรรยากาศและที่นั่ง',
}

aspect_names = list(ASPECT_PROMPTS.keys())
aspect_vecs = e5_model.encode(list(ASPECT_PROMPTS.values()), normalize_embeddings=True)

doc_sim = X_e5_doc @ aspect_vecs.T
clause_max_sim = np.vstack([(vecs @ aspect_vecs.T).max(axis=0) for vecs in clause_vecs_per_review])

THRESH = 0.82
rows_r3 = []
for i, r in df.iterrows():
    text = str(r['review_body'])
    for j, asp in enumerate(aspect_names):
        rows_r3.append({
            'review_id': r['review_id'],
            'sentiment': r['sentiment'],
            'aspect': asp,
            'kw_hit': int(any(kw in text for kw in ASPECT_KEYWORDS[asp])),
            'doc_e5_hit': int(doc_sim[i, j] >= THRESH),
            'clause_e5_hit': int(clause_max_sim[i, j] >= THRESH),
            'clause_sim': round(float(clause_max_sim[i, j]), 3),
            'review_body': text,
        })

asp_df = pd.DataFrame(rows_r3)
summary_asp = asp_df.groupby('aspect')[['kw_hit', 'doc_e5_hit', 'clause_e5_hit']].mean().round(3)
summary_asp.columns = ['Keyword_Rate', 'Doc_E5_Rate', 'Clause_Max_E5_Rate']
display(summary_asp)

fig, ax = plt.subplots(figsize=(9.5, 4.5))
x = np.arange(len(aspect_names))
w = 0.25
for i, (col_name, color) in enumerate(zip(['Keyword_Rate', 'Doc_E5_Rate', 'Clause_Max_E5_Rate'], ['#28537A', '#E07A5F', '#2A9D8F'])):
    vals = [summary_asp.loc[a, col_name] for a in aspect_names]
    bars = ax.bar(x + (i - 1) * w, vals, width=w, label=col_name, color=color, edgecolor='black', linewidth=0.6)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(aspect_names, fontsize=9.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Detection Rate', fontweight='bold')
ax.set_title('RQ3: Aspect Detection Dilution Fix (Doc-Level E5 vs. Clause-Max E5)', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig3_rq3_clause_aspect_fix.png', dpi=180, bbox_inches='tight')
plt.show()

indirect = asp_df[(asp_df['kw_hit'] == 0) & (asp_df['clause_e5_hit'] == 1)].sort_values('clause_sim', ascending=False)
print(f'Indirect detections (Keyword=0, Clause-Max E5=1): {len(indirect)} cases')
display(indirect[['review_id', 'sentiment', 'aspect', 'clause_sim', 'review_body']].head(8))
